# **Coleta de dados**

In [1]:
#Bases de Dados 
from pathlib import Path
import pandas as pd
#import duckdb

# Visualização
import plotly.express as px

# Modelagem 
import numpy as np 
from typing import List, Optional

In [2]:
# Configurações de exibição
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 100
%config SqlMagic.named_parameters = "enabled"
%config SqlMagic.autocommit = True

# Se preferir memória: 
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [3]:
#path = Path("/home/carolina/Área de trabalho/vigimed/data/03_gold")
path = Path(r"C:\Users\silma\Projetos\vigimed\data\03_gold")
# Caminhos completos para o DuckDB (parâmetros nomeados não expandem expressões)
path_atc = str(path / 'dim_atc/dim_atc.parquet')
path_fat_medicamentos = str(path / 'fat_medicamentos/fat_medicamentos.parquet')
path_dim_notificacoes = str(path / 'dim_notificacoes/dim_notificacoes.parquet')
path_fat_reacoes = str(path / 'fat_reacoes/fat_reacoes.parquet')
path_dim_soc_llt = str(path / 'dim_soc_llt/dim_soc_llt.parquet')

In [4]:
%sql ROLLBACK;

Running query in 'duckdb:///:memory:'

,Success


In [5]:
%%sql
DROP TABLE IF EXISTS dim_atc;
CREATE TABLE dim_atc AS
SELECT * FROM read_parquet(:path_atc);

DROP TABLE IF EXISTS fat_medicamentos;
CREATE TABLE fat_medicamentos AS
SELECT *
FROM read_parquet(:path_fat_medicamentos);

DROP TABLE IF EXISTS dim_notificacoes;
CREATE TABLE dim_notificacoes AS
SELECT *
FROM read_parquet(:path_dim_notificacoes);

DROP TABLE IF EXISTS fat_reacoes;
CREATE TABLE fat_reacoes AS
SELECT *
FROM read_parquet(:path_fat_reacoes);

DROP TABLE IF EXISTS dim_soc_llt;
CREATE TABLE dim_soc_llt AS
SELECT *
FROM read_parquet(:path_dim_soc_llt);
 

Running query in 'duckdb:///:memory:'

,Success


# **5️⃣ Entendimento dos dados**

### dim_notificacoes

In [6]:
%%sql 
select * from dim_notificacoes limit 2

Running query in 'duckdb:///:memory:'

,IDENTIFICACAO_NOTIFICACAO,UF_CHAVE,UF_VALOR,TIPO_ENTRADA_VIGIMED_CHAVE,TIPO_ENTRADA_VIGIMED_VALOR,RECEBIDO_DE_CHAVE,RECEBIDO_DE_VALOR,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO_CHAVE,TIPO_NOTIFICACAO_VALOR,NOTIFICACAO_PARENT_CHILD_CHAVE,NOTIFICACAO_PARENT_CHILD_VALOR,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO_TIPO_CHAVE,IDADE_MOMENTO_REACAO_TIPO_VALOR,IDADE_MOMENTO_REACAO_VALOR,GRUPO_IDADE_CHAVE,GRUPO_IDADE_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR,SEXO_CHAVE,SEXO_VALOR,GESTANTE_CHAVE,GESTANTE_VALOR,LACTANTE_CHAVE,LACTANTE_VALOR,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE_CHAVE,GRAVE_VALOR,GRAVIDADE_CHAVE,GRAVIDADE_VALOR,DESFECHO_CHAVE,DESFECHO_VALOR,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,RELACAO_MEDICAMENTO_EVENTO_CHAVE,RELACAO_MEDICAMENTO_EVENTO_VALOR,NOME_MEDICAMENTO_WHODRUG,ACAO_ADOTADA_CHAVE,ACAO_ADOTADA_VALOR,NOTIFICADOR_CHAVE,NOTIFICADOR_VALOR,ATUALIZADO,HASH_BRONZE,HASH_SILVER,HASH_GOLD
0,BR-ANVISA-300212656,25,SP,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,2023-09-28,2023-09-28,NaT,1,NOTIFICACAO ESPONTANEA,0,DESCONHECIDO,1990-01-31,1,ANO,30.0,0,DESCONHECIDO,0,DESCONHECIDO,NaN,1,FEMININO,2,NĂO,2,NĂO,68.0,165,Hemiparesia,1,SIM,1,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,1,RECUPERADO,2021-01-25,2021-02-06,4,DIA,12.0,2,CONCOMITANTE,Tamisa,2,NAO APLICAVEL,4,MEDICO,2026-01-15,c540f6f2873cded948f9a6156d6a92cf9f59b7e110f84e...,3c04663c8c6a69bba2510ab19fa043d1d6670b81e42857...,95ef1ec6b9f7b04d5d423f561ef99f6fe7ea4ce5f916a9...
1,BR-ANVISA-300208322,25,SP,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,2023-09-01,2023-09-01,NaT,1,NOTIFICACAO ESPONTANEA,0,DESCONHECIDO,NaT,0,DESCONHECIDO,NaN,0,DESCONHECIDO,0,DESCONHECIDO,NaN,1,FEMININO,2,NĂO,2,NĂO,None,None,Cefaleia,2,NĂO,1,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,0,DESCONHECIDO,2021-01-22,NaT,0,DESCONHECIDO,NaN,1,SUSPEITO,CoronaVac,2,NAO APLICAVEL,2,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,2026-01-15,1f6805d588f843fe6f27dc43ae2026701e8d0400f3a101...,716905c5a2cc460902e5682c94bd1379f674208b0c6ef5...,8db30198a3122b4ee42039e05a0d958d0caa1efd8f2ece...


In [7]:
%%sql 
select IDENTIFICACAO_NOTIFICACAO,count(1) as qt
from dim_notificacoes 
group by 1
having qt > 1
limit 10

Running query in 'duckdb:///:memory:'

,IDENTIFICACAO_NOTIFICACAO,qt


In [8]:
%%sql 
select * from dim_notificacoes 
where IDENTIFICACAO_NOTIFICACAO = 'BR-ANVISA-300171130'

Running query in 'duckdb:///:memory:'

,IDENTIFICACAO_NOTIFICACAO,UF_CHAVE,UF_VALOR,TIPO_ENTRADA_VIGIMED_CHAVE,TIPO_ENTRADA_VIGIMED_VALOR,RECEBIDO_DE_CHAVE,RECEBIDO_DE_VALOR,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO_CHAVE,TIPO_NOTIFICACAO_VALOR,NOTIFICACAO_PARENT_CHILD_CHAVE,NOTIFICACAO_PARENT_CHILD_VALOR,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO_TIPO_CHAVE,IDADE_MOMENTO_REACAO_TIPO_VALOR,IDADE_MOMENTO_REACAO_VALOR,GRUPO_IDADE_CHAVE,GRUPO_IDADE_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR,SEXO_CHAVE,SEXO_VALOR,GESTANTE_CHAVE,GESTANTE_VALOR,LACTANTE_CHAVE,LACTANTE_VALOR,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE_CHAVE,GRAVE_VALOR,GRAVIDADE_CHAVE,GRAVIDADE_VALOR,DESFECHO_CHAVE,DESFECHO_VALOR,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,RELACAO_MEDICAMENTO_EVENTO_CHAVE,RELACAO_MEDICAMENTO_EVENTO_VALOR,NOME_MEDICAMENTO_WHODRUG,ACAO_ADOTADA_CHAVE,ACAO_ADOTADA_VALOR,NOTIFICADOR_CHAVE,NOTIFICADOR_VALOR,ATUALIZADO,HASH_BRONZE,HASH_SILVER,HASH_GOLD
0,BR-ANVISA-300171130,25,SP,1,SERVICOS DE SAUDE,4,CENTRO REGIONAL DE FARMACOVIGILANCIA,2023-01-10,2023-01-10,2023-01-10,1,NOTIFICACAO ESPONTANEA,0,DESCONHECIDO,NaT,1,ANO,58.0,6,ADULTO,0,DESCONHECIDO,NaN,2,MASCULINO,0,DESCONHECIDO,0,DESCONHECIDO,None,None,Prurido oral,2,NĂO,0,DESCONHECIDO,1,RECUPERADO,2023-01-10,2023-01-10,0,DESCONHECIDO,NaN,1,SUSPEITO,Henetix,0,DESCONHECIDO,3,FARMACEUTICO,2026-01-15,0a98b73bf675dcce6bfbe0a7e08333c2bd6dad4a7ea962...,6c94238fcb1543d2c24c3036169c67658af91aa5c58794...,c1a24670cc84b3b40cdfaa01d5fa0f7b5a8b75b7113813...


### fat_medicamentos

In [9]:
%%sql 
select * from fat_medicamentos 
where IDENTIFICACAO_NOTIFICACAO = 'BR-ANVISA-300171130'

Running query in 'duckdb:///:memory:'

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO_CHAVE,RELACAO_MEDICAMENTO_EVENTO_VALOR,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,ATC_CODE_LEVEL_4,DETENTOR_REGISTRO,DETENTOR_REGISTRO_SCORE,DETENTOR_REGISTRO_CHAVE,CONCENTRACAO_TIPO_CHAVE,CONCENTRACAO_TIPO_VALOR,CONCENTRACAO_VALOR,CONCENTRACAO_VALOR_NUMERADOR,CONCENTRACAO_VALOR_DENOMINADOR,COMPONENTE_SUSPEITO_CHAVE,COMPONENTE_SUSPEITO_VALOR,ACAO_ADOTADA_CHAVE,ACAO_ADOTADA_VALOR,PROB_ADIC_ABUSO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO,INDICACAO_MEDDRA_REACAO_CHAVE,INDICACAO_RELATADA_NOTIFICADOR_INICIAL_REACAO_CHAVE,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR,DOSE_VALOR,FREQUENCIA_DOSE,POSOLOGIA,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA_CHAVE,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_MAE_PAI_CHAVE,NUMELO_LOTE,ATUALIZADO,HASH_BRONZE,HASH_SILVER,HASH_GOLD
0,BR-ANVISA-300171130,2,CONCOMITANTE,NOVALGINA,METAMIZOLE SODIUM,N02BB,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,0,DESCONHECIDO,0,0,0,0,0,0,0,0,0,0,0,0,0,1,MILLIGRAM (MG),1000.0,,None,0,DESCONHECIDO,NaN,2023-01-10,2023-01-10,14,6,5,None,2026-01-15,E8421CD05C810683B262026252DF85A8CB234025058ECE...,4fe24e5960a66d6cb4e5e3d2bd541f87c67a0fe5f0277e...,563044c760ddaa95c1b5702fa524f2a7fa6e4cd6ce7727...
1,BR-ANVISA-300171130,2,CONCOMITANTE,BUSCOPAN,HYOSCINE BUTYLBROMIDE,A03BB,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,0,DESCONHECIDO,0,0,0,0,0,0,0,0,0,0,0,0,0,1,MILLIGRAM (MG),20.0,,None,0,DESCONHECIDO,NaN,2023-01-10,2023-01-10,14,6,5,None,2026-01-15,4AD3B12C405DB5047BDAF8B3178C11665390EA4407B788...,139f988aca4ad7eaee373570884c72c013dace00429f0f...,ee58bd6ff6ea48412f6679644cc02aea44cf969c5304b3...
2,BR-ANVISA-300171130,1,SUSPEITO,HENETIX,IOBITRIDOL,V08AB,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,0,DESCONHECIDO,0,0,0,0,0,0,0,0,0,0,0,0,0,2,MILLILITRE (ML),106.0,,None,0,DESCONHECIDO,NaN,2023-01-10,2023-01-10,16,6,5,22WC366B,2026-01-15,C2F0D1941A6EB562A55B581E0F847EAB15244D0E276E9A...,b389d17b8157fba4987af824119c2978d628a1d4388b0c...,b1939b1e751dc46084949b02b00ccafbf82345fa41cd6f...


### fat_reacoes

In [10]:
%%sql 
select * from fat_reacoes limit 2

Running query in 'duckdb:///:memory:'

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,REACAO_CHAVE,GRAVE_CHAVE,GRAVE_VALOR,DESFECHO_CHAVE,DESFECHO_VALOR,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA,ATUALIZADO,HASH_BRONZE,HASH_SILVER,HASH_GOLD
0,BR-ANVISA-300000004,NaT,NaT,SOC_10040785_HLGT_10014982_HLT_10049293_PT_100...,2,Não,3,Recuperado/Resolvido,4,DIA,3.0,0,0,0,0,0,0,2026-01-15,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0b...,8c2bd3985ba5d5c3c38c928ced3769124a9f33ef2ff3f1...,31c44de9604bad343edd29f25f1317be500fc6836be77b...
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,SOC_10015919_HLGT_10015917_HLT_10030032_PT_100...,1,Sim,3,Recuperado/Resolvido,0,DESCONHECIDO,NaN,0,0,0,0,1,0,2026-01-15,21a5c47062a6ba174af4058e70e681ff97ba91db760676...,056c972fd2c77ca0bfdae5b5950379182d491a356c52dc...,a270a06c9ee9c3fc99b9f5b794a28c5768943f93594ab7...


In [11]:
%%sql 
select * from fat_reacoes 
left join dim_soc_llt using (REACAO_CHAVE)
where IDENTIFICACAO_NOTIFICACAO = 'BR-ANVISA-300171130'

Running query in 'duckdb:///:memory:'

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,REACAO_CHAVE,GRAVE_CHAVE,GRAVE_VALOR,DESFECHO_CHAVE,DESFECHO_VALOR,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA,ATUALIZADO,HASH_BRONZE,HASH_SILVER,HASH_GOLD,PRIMARY_FLAG,SOC_CODE,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_NAME,HLT_CODE,HLT_NAME,PT_CODE,PT_NAME,LLT_CODE,LLT_NAME
0,BR-ANVISA-300171130,2023-01-10,2023-01-10,SOC_10017947_HLGT_10031013_HLT_10031021_PT_100...,2,Não,3,Recuperado/Resolvido,0,DESCONHECIDO,NaN,0,0,0,0,0,0,2026-01-15,bc80540a7cbefd2aca595577b9eefb0b73bc6ecac48230...,7d21ef0b502c7bd709fc861d93e0c85ca914102ce072fe...,f9391ca3f8e4727802351af46704d4dd3c54cfe25be672...,Y,10017947,Distúrbios gastrointestinais,Gastr,10017947,10031013,Quadros clínicos de tecido mole oral,10031021,Sinais e sintomas em tecido mole oral,10052894,Prurido oral,10052894,Prurido oral
1,BR-ANVISA-300171130,2023-01-10,2023-01-10,SOC_10038738_HLGT_10046304_HLT_10028736_PT_100...,2,Não,3,Recuperado/Resolvido,0,DESCONHECIDO,NaN,0,0,0,0,0,0,2026-01-15,33cda90c040b2714862d636436c2f0105c1307a9b593d5...,2b5248a44f91e424536763f2b5b8b5bf7c487473ccf6cf...,9bd9cc8cca01201eaa92960d4019701a573d2a08d4bbee...,Y,10038738,"Distúrbios respiratórios, torácicos e do media...",Resp,10038738,10046304,Distúrbios das vias aéreas superiores (excl. i...,10028736,Inflamações e congestão nasais,10028735,Congestão nasal,10028735,Congestão nasal
2,BR-ANVISA-300171130,2023-01-10,2023-01-10,SOC_10038738_HLGT_10079101_HLT_10046313_PT_100...,2,Não,3,Recuperado/Resolvido,0,DESCONHECIDO,NaN,0,0,0,0,0,0,2026-01-15,8d89edb6a4354411417b35144154c636741cce9c0dc40f...,39211221f089be0ed8d56477b01240c96ba18fb8112d9c...,515bec7e4db47ec59691c1dd6830483334b2f8055a23d9...,Y,10038738,"Distúrbios respiratórios, torácicos e do media...",Resp,10038738,10079101,Sinais e sintomas do trato respiratório,10046313,Sinais e sintomas das vias aéreas superiores,10041232,Espirros,10041233,Espirros excessivos


### dim_soc_llt

In [12]:
%%sql 
select * from dim_soc_llt limit 2

Running query in 'duckdb:///:memory:'

,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_NAME,HLT_CODE,HLT_NAME,PT_CODE,PT_NAME,LLT_CODE,LLT_NAME
0,SOC_10005329_HLGT_10002086_HLT_10002042_PT_100...,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002043,Anemia por deficiência de folato
1,SOC_10005329_HLGT_10002086_HLT_10002042_PT_100...,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002044,Anemia por deficiência de ácido fólico


In [13]:
%%sql 
select COUNT(1),COUNT(DISTINCT REACAO_CHAVE) from dim_soc_llt limit 2

Running query in 'duckdb:///:memory:'

,count(1),count(DISTINCT REACAO_CHAVE)
0,142785,142785


### 6.1 Aplicacao de regras de negócio

In [21]:
%%sql
DROP TABLE IF EXISTS fat_analytics;

CREATE TABLE fat_analytics AS
WITH fat_med AS (
    SELECT
        m.IDENTIFICACAO_NOTIFICACAO,
        atc.ATC_CODE_LEVEL_4_LEVEL_NAME,
        CASE
            WHEN atc.ATC_CODE_LEVEL_4 IN ('L04AB', 'L04AA', 'L01FA', 'L04AC', 'L04AF') THEN
                CASE
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%rituximab%'   THEN 'L01FA01_Rituximabe'
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%abatacept%'   THEN 'L04AA24_Abatacepte'
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%etanercept%'  THEN 'L04AB01_Etanercepte'
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%infliximab%'  THEN 'L04AB02_Infliximabe'
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%adalimumab%'  THEN 'L04AB04_Adalimumabe'
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab%' THEN 'L04AB05_Certolizumabe Pegol'
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%golimumab%'   THEN 'L04AB06_Golimumabe'
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%tocilizumab%' THEN 'L04AC07_Tocilizumabe'
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%tofacitinib%' THEN 'L04AF01_Tofacitinibe'
                    WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%baricitinib%' THEN 'L04AF02_Baricitinibe'
                    ELSE 'Outros'
                END
            ELSE 'Outros'
        END AS ATC_LEVEL_5
    FROM fat_medicamentos m
    INNER JOIN dim_atc AS atc
        ON m.ATC_CODE_LEVEL_4 = atc.ATC_CODE_LEVEL_4
       AND atc.ATC_LEVEL = 4
),
fat AS (
    SELECT
        strftime(TRY_CAST(dim.DATA_INCLUSAO_SISTEMA AS DATE), '%Y-%m') AS DATA_NOTIFICACAO,
        dim.IDENTIFICACAO_NOTIFICACAO AS ID,
        dim.UF_VALOR AS UF,
        dim.SEXO_VALOR AS SEXO,
        fat_med.ATC_LEVEL_5,
        ds.LLT_NAME AS REACAO_LLT,
        ds.PT_NAME AS REACAO_PT
    FROM dim_notificacoes AS dim
    INNER JOIN fat_med
        ON dim.IDENTIFICACAO_NOTIFICACAO = fat_med.IDENTIFICACAO_NOTIFICACAO
    INNER JOIN fat_reacoes fr
        ON dim.IDENTIFICACAO_NOTIFICACAO = fr.IDENTIFICACAO_NOTIFICACAO
    LEFT JOIN dim_soc_llt ds
        ON ds.REACAO_CHAVE = fr.REACAO_CHAVE
)
SELECT DISTINCT *
FROM (
    SELECT *
    FROM fat
    WHERE ATC_LEVEL_5 <> 'Outros'

    UNION ALL

    SELECT *
    FROM fat AS a
    WHERE a.ATC_LEVEL_5 = 'Outros'
      AND a.ID NOT IN (
          SELECT b.ID
          FROM fat AS b
          WHERE b.ATC_LEVEL_5 <> 'Outros'
      )
);

Running query in 'duckdb:///:memory:'

,Success


### 6.2 Checagens

In [22]:
%%sql
select * from fat_analytics limit 5

Running query in 'duckdb:///:memory:'

,DATA_NOTIFICACAO,ID,UF,SEXO,ATC_LEVEL_5,REACAO_LLT,REACAO_PT
0,2021-06,BR-ANVISA-300061333,DESCONHECIDO,FEMININO,L04AB06_Golimumabe,Infecção do trato urinário,Infecção do trato urinário
1,2021-06,BR-ANVISA-300061436,DESCONHECIDO,MASCULINO,L04AB04_Adalimumabe,Evento adverso,Evento adverso
2,2025-06,BR-ANVISA-300341192,SP,DESCONHECIDO,L04AB02_Infliximabe,Irritação da garganta,Irritação da garganta
3,2025-06,BR-ANVISA-300341263,PE,FEMININO,L04AB06_Golimumabe,Dor de cabeça,Cefaleia
4,2025-06,BR-ANVISA-300341328,SP,MASCULINO,L04AB06_Golimumabe,Iridociclite,Iridociclite


In [24]:
%%sql
select ATC_LEVEL_5,count(*) as qt,count(distinct ID) as qt_notificacoes from fat_analytics group by ATC_LEVEL_5 

Running query in 'duckdb:///:memory:'

,ATC_LEVEL_5,qt,qt_notificacoes
0,L04AB05_Certolizumabe Pegol,4508,1212
1,L04AF01_Tofacitinibe,423,143
2,Outros,718832,285081
3,L04AB02_Infliximabe,35102,11167
4,L04AB06_Golimumabe,10911,3994
5,L04AB04_Adalimumabe,7583,1588
6,L04AA24_Abatacepte,598,172
7,L04AF02_Baricitinibe,548,208
8,L01FA01_Rituximabe,8331,3281
9,L04AB01_Etanercepte,2070,369


In [25]:
%%sql
select id,count(1) from fat_analytics group by 1 having count(1) > 1 limit 2

Running query in 'duckdb:///:memory:'

,ID,count(1)
0,BR-ANVISA-300169373,2
1,BR-ANVISA-300178116,3


In [26]:
%%sql 
select * from fat_analytics 
where id ='BR-ANVISA-300315256'

Running query in 'duckdb:///:memory:'

,DATA_NOTIFICACAO,ID,UF,SEXO,ATC_LEVEL_5,REACAO_LLT,REACAO_PT
0,2025-01,BR-ANVISA-300315256,SP,FEMININO,Outros,Doença pulmonar obstrutiva crônica,Doença pulmonar obstrutiva crônica
1,2025-01,BR-ANVISA-300315256,SP,FEMININO,Outros,Uso não descrito em bula (off label),Uso não descrito em bula (off label)


### 6.3 Persistindo a Base

In [27]:
%%sql df <<
select * from fat_analytics

Running query in 'duckdb:///:memory:'

In [28]:
df.to_parquet(path/'fat_analytical/fat_analytical.parquet', index=False)